In [2]:
import pandas as pd

llm_candidates = pd.read_pickle("/content/llm_candidates.pkl")

import json
with open("/content/jd_structured.json", "r") as f:
    jd = json.load(f)

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
top_200=llm_candidates[:200]

In [ ]:
PAIRWISE_PROMPT = """
You are hiring a Senior Applied ML Engineer.

Focus on:

- Search
- Retrieval
- Ranking
- Recommendation Systems
- Production ML Engineering

Return ONLY JSON:

{
    "winner":"A",
    "confidence":0.85
}

winner must be A or B.
confidence must be between 0.5 and 1.0.
"""

In [ ]:
def compare_candidates(row_a, row_b):

    prompt = f"""
Candidate A:

{row_a['profile'][:2500]}

Signals:
jd_skill_score={row_a['jd_skill_score']}
behavior_score={row_a['behavior_score']}

----------------------------------

Candidate B:

{row_b['profile'][:2500]}

Signals:
jd_skill_score={row_b['jd_skill_score']}
behavior_score={row_b['behavior_score']}
"""

    messages = [
        {"role": "system", "content": PAIRWISE_PROMPT},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError(text)

    return json.loads(match.group())

In [ ]:
from collections import defaultdict

In [6]:

import json
import re
import torch
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
wins = defaultdict(float)

N = len(top_200)

for i in tqdm(range(N)):

    for offset in [1,2,3]:

        j = i + offset

        if j >= N:
            continue

        row_a = top_200.iloc[i]
        row_b = top_200.iloc[j]

        try:

            result = compare_candidates(
                row_a,
                row_b
            )

            winner = result["winner"]
            confidence = float(
                result["confidence"]
            )

            if winner == "A":

                wins[
                    row_a["candidate_id"]
                ] += confidence

            else:

                wins[
                    row_b["candidate_id"]
                ] += confidence

        except Exception as e:

            print(e)

NameError: name 'defaultdict' is not defined

In [ ]:
pairwise_df = pd.DataFrame({
    "candidate_id": list(wins.keys()),
    "pairwise_score": list(wins.values())
})

In [ ]:
top_200 = top_200.merge(
    pairwise_df,
    on="candidate_id",
    how="left"
)

top_200["pairwise_score"] = (
    top_200["pairwise_score"]
    .fillna(0)
)

In [ ]:
top_150 = (
    top_200
    .sort_values(
        "pairwise_score",
        ascending=False
    )
    .head(150)
    .reset_index(drop=True)
)

In [ ]:
top_150

,candidate_id,profile,jd_skill_score,behavior_score,pairwise_score
0,CAND_0050876,Candidate Summary:\nMachine learning engineer ...,79.91,63.16,5.10
1,CAND_0075439,Candidate Summary:\nMachine learning engineer ...,79.60,65.32,5.10
2,CAND_0020877,Candidate Summary:\nMachine learning engineer ...,82.64,68.04,4.35
3,CAND_0065277,Candidate Summary:\nData scientist / ML engine...,77.07,66.40,4.30
4,CAND_0042506,Candidate Summary:\nMachine learning engineer ...,86.07,74.76,4.25
...,...,...,...,...,...
145,CAND_0044883,Candidate Summary:\nMachine learning engineer ...,84.24,57.16,1.75
146,CAND_0070398,Candidate Summary:\nMachine learning engineer ...,83.54,66.28,1.75
147,CAND_0096142,Candidate Summary:\nMachine learning engineer ...,86.66,64.48,1.75
148,CAND_0026532,Candidate Summary:\nMachine learning engineer ...,82.80,67.16,1.75


In [7]:

import json
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
top_150 = pd.read_pickle(
    "top_150.pkl"
)
SCORING_PROMPT = """
You are an expert recruiter.

Evaluate the candidate.

Return ONLY JSON:

{
  "overall_score": 0,
  "reasoning": ""
}

overall_score must be between 0 and 100.

reasoning must be 2-3 concise sentences.

Use only evidence explicitly present in the profile.
"""

In [8]:
def evaluate_candidate(row):

    prompt = f"""
Candidate:

{row['profile'][:3500]}

Signals:

jd_skill_score={row['jd_skill_score']}
behavior_score={row['behavior_score']}
"""

    messages = [
        {"role":"system",
         "content":SCORING_PROMPT},
        {"role":"user",
         "content":prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if match:
        return json.loads(match.group())

    return {
        "overall_score": 0,
        "reasoning": text[:300]
    }

In [9]:
results = []

for _, row in tqdm(
    top_150.iterrows(),
    total=len(top_150)
):

    try:

        result = evaluate_candidate(
            row
        )

    except Exception as e:

        result = {
            "overall_score": 0,
            "reasoning": str(e)
        }

    results.append({
        "candidate_id":
            row["candidate_id"],

        "llm_score":
            result["overall_score"],

        "reasoning":
            result["reasoning"],



        "pairwise_score":
            row["pairwise_score"]
    })

  0%|          | 0/150 [00:00<?, ?it/s]

In [10]:
final_df = pd.DataFrame(results)

final_df["final_score"] = (
      0.70 * final_df["llm_score"]
    + 0.30 * final_df["pairwise_score"]

)

final_top_100 = (
    final_df
    .sort_values(
        "final_score",
        ascending=False
    )
    .head(100)
    .reset_index(drop=True)
)

In [12]:
display(
    final_top_100[
        [
            "candidate_id",
            "final_score",
            "reasoning"
        ]
    ]
)

,candidate_id,final_score,reasoning
0,CAND_0099806,67.265,The candidate demonstrates strong technical sk...
1,CAND_0042029,64.275,The candidate demonstrates strong technical sk...
2,CAND_0028793,64.035,The candidate demonstrates strong expertise in...
3,CAND_0030348,64.035,The candidate demonstrates strong expertise in...
4,CAND_0006418,64.020,The candidate demonstrates strong technical sk...
...,...,...,...
95,CAND_0031593,60.265,The candidate has extensive experience in mach...
96,CAND_0081563,60.170,The candidate has extensive experience as both...
97,CAND_0017093,60.170,The candidate has significant experience in pr...
98,CAND_0039383,60.075,The candidate demonstrates strong expertise in...


In [15]:
final_top_100.to_pickle(
    "final_top_100.pkl"
)

In [16]:
from google.colab import files

files.download("final_top_100.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import pandas as pd

# Load your final dataframe
# If already in memory, skip this line
# final_top_100 = pd.read_pickle("final_top_100.pkl")

# Sort according to challenge rules:
# 1. Score descending
# 2. candidate_id ascending for tie-breaks
submission = (
    final_top_100
    .sort_values(
        ["final_score", "candidate_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# Add ranks 1-100
submission["rank"] = range(1, len(submission) + 1)

# Rename columns to required format
submission = submission.rename(
    columns={"final_score": "score"}
)

# Keep only required columns in required order
submission = submission[
    ["candidate_id", "rank", "score", "reasoning"]
]

# Save CSV
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"\nSaved {len(submission)} rows to submission.csv")

   candidate_id  rank   score  \
0  CAND_0099806     1  67.265   
1  CAND_0042029     2  64.275   
2  CAND_0028793     3  64.035   
3  CAND_0030348     4  64.035   
4  CAND_0006418     5  64.020   

                                           reasoning  
0  The candidate demonstrates strong technical sk...  
1  The candidate demonstrates strong technical sk...  
2  The candidate demonstrates strong expertise in...  
3  The candidate demonstrates strong expertise in...  
4  The candidate demonstrates strong technical sk...  

Saved 100 rows to submission.csv


In [18]:
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>